In [2]:
# ============================================================
# XGBOOST - COMPARACIÓN DE PREPROCESAMIENTO
# normal vs stemming vs lematización
# Variantes: MX / ES / CU
# Métrica principal: F1-Macro
# Validación: Stratified 5-Fold
# ============================================================

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

from xgboost import XGBClassifier


# ============================================================
# 1. CONFIGURACIÓN GENERAL
# ============================================================

RANDOM_STATE = 42

VARIANTES = ['mx', 'es', 'cu']

PREPROCESAMIENTOS = {
    'normal': '',
    'stem': '_stem',
    'lemma': '_lemma'
}

FEATURE_COLS = [
    'n_exc',  # exclamaciones
    'n_int',  # interrogaciones
    'n_may',  # palabras en mayúsculas
    'n_emo',  # emojis
    'n_ris',  # expresiones de risa
    'n_neg',  # negaciones
    'n_elo',  # elongaciones
    'n_com',  # comillas
    'n_pun'   # puntos suspensivos
]

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 2. CARGA DE DATOS
# ============================================================
#
# Modifica DATA_DIR si tus CSV están en otra carpeta.
#
# Se espera una estructura parecida a:
#
# train_clean_mx.csv
# train_clean_es.csv
# train_clean_cu.csv
#
# train_clean_stem_mx.csv
# train_clean_stem_es.csv
# train_clean_stem_cu.csv
#
# train_clean_lemma_mx.csv
# train_clean_lemma_es.csv
# train_clean_lemma_cu.csv
#
# ============================================================

DATA_DIR = '../data'


def cargar_train(variante, prep):

    sufijo = PREPROCESAMIENTOS[prep]

    ruta = f'{DATA_DIR}/train_clean{sufijo}_{variante}.csv'

    print(f'Cargando: {ruta}')

    df = pd.read_csv(ruta)

    return df


# ============================================================
# 3. PREPROCESADOR
# ============================================================
#
# TF-IDF:
# - unigramas + bigramas
# - hasta 20 000 términos
# - min_df=2 para eliminar términos extremadamente raros
# - sublinear_tf=True
#
# NO eliminamos stopwords aquí.
#
# TF-IDF acepta ngram_range, min_df, max_df,
# max_features y sublinear_tf de forma nativa.
#
# ============================================================

def crear_preprocesador():

    tfidf = TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),

        min_df=2,
        max_df=0.98,

        max_features=20000,

        lowercase=False,

        sublinear_tf=True,

        dtype=np.float32
    )

    preprocesador = ColumnTransformer(
        transformers=[
            (
                'tfidf',
                tfidf,
                'MESSAGE_CLEAN'
            ),
            (
                'linguisticas',
                'passthrough',
                FEATURE_COLS
            )
        ],
        remainder='drop'
    )

    return preprocesador


# ============================================================
# 4. XGBOOST BASE
# ============================================================
#
# Esta NO es la configuración final.
# Solo sirve para comparar normal / stem / lemma.
#
# Después optimizaremos hiperparámetros con Optuna.
#
# ============================================================

def crear_xgboost(y):

    negativos = (y == 0).sum()
    positivos = (y == 1).sum()

    scale_pos_weight = negativos / positivos

    modelo = XGBClassifier(

        # Problema binario
        objective='binary:logistic',

        # Boosting
        n_estimators=300,
        learning_rate=0.05,

        # Complejidad de árboles
        max_depth=5,
        min_child_weight=2,

        # Muestreo
        subsample=0.8,
        colsample_bytree=0.8,

        # Regularización
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,

        # Desbalance
        scale_pos_weight=scale_pos_weight,

        # Algoritmo eficiente
        tree_method='hist',

        # Reproducibilidad
        random_state=RANDOM_STATE,

        # Paralelización interna
        n_jobs=-1,

        # Métrica interna de entrenamiento
        eval_metric='logloss'
    )

    return modelo


# ============================================================
# 5. FUNCIÓN DE EVALUACIÓN
# ============================================================

def evaluar_preprocesamiento(variante, prep):

    df = cargar_train(
        variante=variante,
        prep=prep
    )

    # --------------------------------------------------------
    # Comprobaciones
    # --------------------------------------------------------

    columnas_necesarias = (
        ['MESSAGE_CLEAN', 'IS_IRONIC']
        + FEATURE_COLS
    )

    faltantes = [
        col
        for col in columnas_necesarias
        if col not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f'Faltan columnas en {variante}-{prep}: '
            f'{faltantes}'
        )

    # --------------------------------------------------------
    # Datos
    # --------------------------------------------------------

    X = df[
        ['MESSAGE_CLEAN'] + FEATURE_COLS
    ].copy()

    y = df['IS_IRONIC'].astype(int)

    # Por si hubiera NaN en texto
    X['MESSAGE_CLEAN'] = (
        X['MESSAGE_CLEAN']
        .fillna('')
        .astype(str)
    )

    # Por si hubiera NaN en features lingüísticas
    X[FEATURE_COLS] = (
        X[FEATURE_COLS]
        .fillna(0)
    )

    # --------------------------------------------------------
    # Información de clases
    # --------------------------------------------------------

    negativos = (y == 0).sum()
    positivos = (y == 1).sum()

    print(
        f'Clase 0: {negativos} | '
        f'Clase 1: {positivos} | '
        f'ratio: {negativos / positivos:.3f}'
    )

    # --------------------------------------------------------
    # Pipeline
    # --------------------------------------------------------

    pipeline = Pipeline(
        steps=[
            (
                'features',
                crear_preprocesador()
            ),
            (
                'xgb',
                crear_xgboost(y)
            )
        ]
    )

    # --------------------------------------------------------
    # Cross Validation
    # --------------------------------------------------------

    scores = cross_validate(
        estimator=pipeline,

        X=X,
        y=y,

        cv=CV,

        scoring={
            'f1_macro': 'f1_macro',
            'accuracy': 'accuracy',
            'precision_macro': 'precision_macro',
            'recall_macro': 'recall_macro'
        },

        # Evitamos paralelizar folds + XGBoost simultáneamente
        n_jobs=1,

        return_train_score=False
    )

    # --------------------------------------------------------
    # Resultado
    # --------------------------------------------------------

    resultado = {
        'variante': variante,
        'preprocesamiento': prep,

        'f1_macro_mean':
            scores['test_f1_macro'].mean(),

        'f1_macro_std':
            scores['test_f1_macro'].std(),

        'accuracy_mean':
            scores['test_accuracy'].mean(),

        'precision_macro_mean':
            scores['test_precision_macro'].mean(),

        'recall_macro_mean':
            scores['test_recall_macro'].mean()
    }

    return resultado


# ============================================================
# 6. EJECUTAR TODOS LOS EXPERIMENTOS
# ============================================================

resultados = []

for variante in VARIANTES:

    print('\n')
    print('=' * 65)
    print(f'VARIANTE: {variante.upper()}')
    print('=' * 65)

    for prep in PREPROCESAMIENTOS:

        print('\n')
        print('-' * 45)
        print(f'Preprocesamiento: {prep.upper()}')
        print('-' * 45)

        resultado = evaluar_preprocesamiento(
            variante=variante,
            prep=prep
        )

        resultados.append(resultado)

        print(
            '\nResultado: '
            f'F1-Macro = '
            f'{resultado["f1_macro_mean"]:.4f} '
            f'± '
            f'{resultado["f1_macro_std"]:.4f}'
        )


# ============================================================
# 7. TABLA COMPLETA DE RESULTADOS
# ============================================================

df_resultados = pd.DataFrame(resultados)

df_resultados = (
    df_resultados
    .sort_values(
        by=[
            'variante',
            'f1_macro_mean'
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

print('\n')
print('=' * 70)
print('RESULTADOS COMPLETOS')
print('=' * 70)

display(
    df_resultados.style.format({
        'f1_macro_mean': '{:.4f}',
        'f1_macro_std': '{:.4f}',
        'accuracy_mean': '{:.4f}',
        'precision_macro_mean': '{:.4f}',
        'recall_macro_mean': '{:.4f}'
    })
)


# ============================================================
# 8. MEJOR PREPROCESAMIENTO POR VARIANTE
# ============================================================

mejores_preprocesamientos = (
    df_resultados
    .sort_values(
        'f1_macro_mean',
        ascending=False
    )
    .groupby(
        'variante',
        as_index=False
    )
    .first()
)

print('\n')
print('=' * 70)
print('MEJOR PREPROCESAMIENTO POR VARIANTE')
print('=' * 70)

display(
    mejores_preprocesamientos.style.format({
        'f1_macro_mean': '{:.4f}',
        'f1_macro_std': '{:.4f}',
        'accuracy_mean': '{:.4f}',
        'precision_macro_mean': '{:.4f}',
        'recall_macro_mean': '{:.4f}'
    })
)


# ============================================================
# 9. MOSTRAR RANKING POR VARIANTE
# ============================================================

for variante in VARIANTES:

    print('\n')
    print('=' * 60)
    print(f'RANKING {variante.upper()}')
    print('=' * 60)

    ranking = (
        df_resultados[
            df_resultados['variante'] == variante
        ]
        .sort_values(
            'f1_macro_mean',
            ascending=False
        )
        [
            [
                'preprocesamiento',
                'f1_macro_mean',
                'f1_macro_std',
                'accuracy_mean',
                'precision_macro_mean',
                'recall_macro_mean'
            ]
        ]
    )

    display(
        ranking.style.format({
            'f1_macro_mean': '{:.4f}',
            'f1_macro_std': '{:.4f}',
            'accuracy_mean': '{:.4f}',
            'precision_macro_mean': '{:.4f}',
            'recall_macro_mean': '{:.4f}'
        })
    )



VARIANTE: MX


---------------------------------------------
Preprocesamiento: NORMAL
---------------------------------------------
Cargando: ../data/train_clean_mx.csv
Clase 0: 1599 | Clase 1: 800 | ratio: 1.999

Resultado: F1-Macro = 0.6127 ± 0.0112


---------------------------------------------
Preprocesamiento: STEM
---------------------------------------------
Cargando: ../data/train_clean_stem_mx.csv
Clase 0: 1599 | Clase 1: 800 | ratio: 1.999

Resultado: F1-Macro = 0.6084 ± 0.0210


---------------------------------------------
Preprocesamiento: LEMMA
---------------------------------------------
Cargando: ../data/train_clean_lemma_mx.csv
Clase 0: 1599 | Clase 1: 800 | ratio: 1.999

Resultado: F1-Macro = 0.6100 ± 0.0092


VARIANTE: ES


---------------------------------------------
Preprocesamiento: NORMAL
---------------------------------------------
Cargando: ../data/train_clean_es.csv
Clase 0: 1598 | Clase 1: 800 | ratio: 1.998

Resultado: F1-Macro = 0.6902 ± 0.0152


----

,variante,preprocesamiento,f1_macro_mean,f1_macro_std,accuracy_mean,precision_macro_mean,recall_macro_mean
0,cu,stem,0.6581,0.0209,0.7025,0.6629,0.6550
1,cu,normal,0.6578,0.0197,0.7025,0.6624,0.6547
2,cu,lemma,0.6389,0.0136,0.6862,0.6432,0.6362
3,es,lemma,0.7028,0.0188,0.7252,0.6986,0.7127
4,es,stem,0.6985,0.0150,0.7202,0.6949,0.7095
5,es,normal,0.6902,0.0152,0.7106,0.6871,0.7033
6,mx,normal,0.6127,0.0112,0.6348,0.6133,0.6246
7,mx,lemma,0.6100,0.0092,0.6336,0.6100,0.6205
8,mx,stem,0.6084,0.0210,0.6307,0.6090,0.6199




MEJOR PREPROCESAMIENTO POR VARIANTE


,variante,preprocesamiento,f1_macro_mean,f1_macro_std,accuracy_mean,precision_macro_mean,recall_macro_mean
0,cu,stem,0.6581,0.0209,0.7025,0.6629,0.6550
1,es,lemma,0.7028,0.0188,0.7252,0.6986,0.7127
2,mx,normal,0.6127,0.0112,0.6348,0.6133,0.6246




RANKING MX


,preprocesamiento,f1_macro_mean,f1_macro_std,accuracy_mean,precision_macro_mean,recall_macro_mean
6,normal,0.6127,0.0112,0.6348,0.6133,0.6246
7,lemma,0.6100,0.0092,0.6336,0.6100,0.6205
8,stem,0.6084,0.0210,0.6307,0.6090,0.6199




RANKING ES


,preprocesamiento,f1_macro_mean,f1_macro_std,accuracy_mean,precision_macro_mean,recall_macro_mean
3,lemma,0.7028,0.0188,0.7252,0.6986,0.7127
4,stem,0.6985,0.0150,0.7202,0.6949,0.7095
5,normal,0.6902,0.0152,0.7106,0.6871,0.7033




RANKING CU


,preprocesamiento,f1_macro_mean,f1_macro_std,accuracy_mean,precision_macro_mean,recall_macro_mean
0,stem,0.6581,0.0209,0.7025,0.6629,0.6550
1,normal,0.6578,0.0197,0.7025,0.6624,0.6547
2,lemma,0.6389,0.0136,0.6862,0.6432,0.6362


In [3]:
# ============================================================
# 10. OPTIMIZACIÓN DE XGBOOST CON OPTUNA
# ============================================================

import optuna

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier


# ============================================================
# 10.1 PREPROCESAMIENTO GANADOR POR VARIANTE
# ============================================================

MEJOR_PREP = {
    'mx': 'normal',
    'es': 'lemma',
    'cu': 'stem'
}


# ============================================================
# 10.2 FUNCIÓN OBJETIVO PARA OPTUNA
# ============================================================

def objective_xgboost(trial, variante):

    prep = MEJOR_PREP[variante]

    df = cargar_train(
        variante=variante,
        prep=prep
    )

    X = df[
        ['MESSAGE_CLEAN'] + FEATURE_COLS
    ].copy()

    y = df['IS_IRONIC'].astype(int)

    # Seguridad ante NaN
    X['MESSAGE_CLEAN'] = (
        X['MESSAGE_CLEAN']
        .fillna('')
        .astype(str)
    )

    X[FEATURE_COLS] = (
        X[FEATURE_COLS]
        .fillna(0)
    )

    # --------------------------------------------------------
    # Espacio de búsqueda
    # --------------------------------------------------------

    params = {

        'n_estimators': trial.suggest_int(
            'n_estimators',
            100,
            800,
            step=50
        ),

        'max_depth': trial.suggest_int(
            'max_depth',
            2,
            10
        ),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.01,
            0.30,
            log=True
        ),

        'min_child_weight': trial.suggest_int(
            'min_child_weight',
            1,
            10
        ),

        'subsample': trial.suggest_float(
            'subsample',
            0.60,
            1.00
        ),

        'colsample_bytree': trial.suggest_float(
            'colsample_bytree',
            0.50,
            1.00
        ),

        'gamma': trial.suggest_float(
            'gamma',
            0.0,
            5.0
        ),

        'reg_alpha': trial.suggest_float(
            'reg_alpha',
            1e-4,
            10.0,
            log=True
        ),

        'reg_lambda': trial.suggest_float(
            'reg_lambda',
            1e-3,
            20.0,
            log=True
        ),

        'scale_pos_weight': trial.suggest_float(
            'scale_pos_weight',
            1.0,
            3.0
        )
    }

    # --------------------------------------------------------
    # Modelo
    # --------------------------------------------------------

    modelo = XGBClassifier(

        objective='binary:logistic',

        tree_method='hist',

        eval_metric='logloss',

        random_state=RANDOM_STATE,

        n_jobs=-1,

        **params
    )

    # --------------------------------------------------------
    # Pipeline
    # --------------------------------------------------------

    pipeline = Pipeline(
        steps=[
            (
                'features',
                crear_preprocesador()
            ),
            (
                'xgb',
                modelo
            )
        ]
    )

    # --------------------------------------------------------
    # Stratified 5-Fold
    # --------------------------------------------------------

    scores = cross_val_score(
        estimator=pipeline,
        X=X,
        y=y,
        cv=CV,
        scoring='f1_macro',

        # Evita paralelizar folds y XGBoost simultáneamente
        n_jobs=1
    )

    return scores.mean()


# ============================================================
# 10.3 EJECUTAR OPTUNA POR VARIANTE
# ============================================================

N_TRIALS = 50

studies = {}

for variante in ['mx', 'es', 'cu']:

    print('\n')
    print('=' * 70)
    print(f'OPTUNA - VARIANTE {variante.upper()}')
    print(
        f'Preprocesamiento seleccionado: '
        f'{MEJOR_PREP[variante]}'
    )
    print('=' * 70)

    sampler = optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )

    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        study_name=f'xgboost_{variante}'
    )

    study.optimize(
        lambda trial: objective_xgboost(
            trial,
            variante
        ),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    studies[variante] = study

    print('\n')
    print(
        f'Mejor F1-Macro CV: '
        f'{study.best_value:.4f}'
    )

    print('\nMejores hiperparámetros:')

    for parametro, valor in study.best_params.items():
        print(
            f'  {parametro}: {valor}'
        )


# ============================================================
# 10.4 RESUMEN DE RESULTADOS DE OPTUNA
# ============================================================

resumen_optuna = []

for variante, study in studies.items():

    fila = {
        'variante': variante,
        'preprocesamiento':
            MEJOR_PREP[variante],

        'best_f1_macro_cv':
            study.best_value,

        'best_trial':
            study.best_trial.number
    }

    fila.update(
        study.best_params
    )

    resumen_optuna.append(fila)


df_optuna = pd.DataFrame(
    resumen_optuna
)

print('\n')
print('=' * 70)
print('MEJORES RESULTADOS OPTUNA')
print('=' * 70)

display(
    df_optuna.style.format({
        'best_f1_macro_cv': '{:.4f}',
        'learning_rate': '{:.5f}',
        'subsample': '{:.4f}',
        'colsample_bytree': '{:.4f}',
        'gamma': '{:.4f}',
        'reg_alpha': '{:.5f}',
        'reg_lambda': '{:.5f}',
        'scale_pos_weight': '{:.4f}'
    })
)


# ============================================================
# 10.5 HISTORIAL DE TRIALS
# ============================================================

for variante, study in studies.items():

    print('\n')
    print('=' * 70)
    print(f'TOP 10 TRIALS - {variante.upper()}')
    print('=' * 70)

    df_trials = study.trials_dataframe()

    columnas = [
        'number',
        'value'
    ] + [
        col
        for col in df_trials.columns
        if col.startswith('params_')
    ]

    display(
        df_trials[
            columnas
        ]
        .sort_values(
            'value',
            ascending=False
        )
        .head(10)
    )

[I 2026-09-21 16:56:55,725] A new study created in memory with name: xgboost_mx




OPTUNA - VARIANTE MX
Preprocesamiento seleccionado: normal


  0%|          | 0/50 [00:00<?, ?it/s]

Cargando: ../data/train_clean_mx.csv
[I 2026-09-21 16:57:01,311] Trial 0 finished with value: 0.5922952189215975 and parameters: {'n_estimators': 350, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.5779972601681014, 'gamma': 0.2904180608409973, 'reg_alpha': 2.1423021757741068, 'reg_lambda': 0.3849583075868115, 'scale_pos_weight': 2.416145155592091}. Best is trial 0 with value: 0.5922952189215975.
Cargando: ../data/train_clean_mx.csv
[I 2026-09-21 16:57:03,710] Trial 1 finished with value: 0.5879048875235989 and parameters: {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.16967533607196555, 'min_child_weight': 3, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.5917022549267169, 'gamma': 1.5212112147976886, 'reg_alpha': 0.042051564509138675, 'reg_lambda': 0.07207895500630218, 'scale_pos_weight': 1.5824582803960838}. Best is trial 0 with value: 0.5922952189215975.
Cargando: ../data/train_c

[I 2026-09-21 17:01:01,995] A new study created in memory with name: xgboost_es


...

,number,value,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_alpha,params_reg_lambda,params_scale_pos_weight,params_subsample
42,42,0.621768,0.757305,4.566597,0.046809,6,1,550,0.002033,0.441521,1.707239,0.626316
43,43,0.617502,0.772791,3.801343,0.030271,6,2,600,0.030038,0.645385,1.739864,0.670964
18,18,0.615537,0.773250,2.036696,0.061588,4,1,650,0.017141,1.003253,1.758781,0.673158
47,47,0.615258,0.726191,4.984074,0.055319,7,1,700,0.000877,0.723884,1.718858,0.623571
39,39,0.614059,0.696572,4.197266,0.075626,6,1,550,0.043187,0.123131,1.963329,0.640018
44,44,0.613553,0.720087,4.092680,0.051367,7,1,550,0.002593,0.187889,1.841414,0.653529
49,49,0.613354,0.676400,4.498098,0.052994,7,2,750,0.001165,0.153434,1.752527,0.661084
48,48,0.610857,0.723241,4.314048,0.047512,5,1,600,0.001030,1.365863,1.521557,0.648010
45,45,0.609553,0.820137,4.180037,0.061119,6,2,550,0.000589,0.922204,1.623194,0.611632
41,41,0.609113,0.754832,4.672825,0.058445,5,2,600,0.007031,0.538749,2.010855,0.633512




TOP 10 TRIALS - ES


,number,value,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_alpha,params_reg_lambda,params_scale_pos_weight,params_subsample
37,37,0.713170,0.920556,2.671698,0.041120,3,1,650,0.000530,1.330254,1.684812,0.945552
22,22,0.712791,0.961906,2.702431,0.034734,3,2,750,0.000195,6.060735,1.250872,0.962329
44,44,0.709161,0.905830,2.681437,0.053654,4,1,700,0.000500,2.313400,1.462256,0.932274
18,18,0.708481,0.952337,2.023356,0.057537,3,1,650,0.000496,3.215607,1.387930,0.872355
46,46,0.708111,0.991349,1.726325,0.050774,4,2,700,0.000841,5.502342,1.547377,0.911112
30,30,0.708033,0.983079,2.253864,0.058834,4,1,700,0.000319,5.958199,1.302882,0.912997
33,33,0.707517,0.963626,2.961514,0.048899,3,1,750,0.000227,7.281301,1.288120,0.938393
17,17,0.707434,0.929279,2.683116,0.049110,3,1,650,0.000123,1.934842,1.371350,0.950947
15,15,0.707052,0.781049,4.183467,0.018716,3,2,650,0.150438,0.888622,1.602216,0.819686
41,41,0.706341,0.947265,1.317132,0.049496,4,1,700,0.000321,6.087418,1.389691,0.880241




TOP 10 TRIALS - CU


,number,value,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_alpha,params_reg_lambda,params_scale_pos_weight,params_subsample
17,17,0.668364,0.825428,2.796169,0.031548,3,1,450,0.013918,0.288309,1.704181,0.785920
42,42,0.666743,0.862842,2.453716,0.029014,3,1,450,0.002753,0.118419,1.446029,0.809243
40,40,0.666620,0.872937,3.082857,0.023003,5,3,400,0.010726,0.109761,1.529572,0.782911
43,43,0.665199,0.849694,3.034163,0.021119,4,2,450,0.017518,0.294885,1.745953,0.758301
39,39,0.663263,0.913396,3.398485,0.022677,4,2,500,0.004851,0.211197,1.916735,0.768305
48,48,0.662639,0.852774,2.260878,0.031422,4,2,450,0.024005,0.262085,1.707629,0.767720
18,18,0.662401,0.911989,2.214959,0.038874,3,1,500,0.001638,0.887082,1.599981,0.769154
35,35,0.662168,0.869845,2.026760,0.022257,4,1,400,0.002212,0.128045,1.728103,0.728764
28,28,0.661305,0.725206,2.219622,0.018505,4,2,400,0.005791,0.255776,1.963172,0.793558
46,46,0.661087,0.846021,2.822249,0.017776,4,1,450,0.002967,0.265219,1.720500,0.786469


In [4]:
# ============================================================
# 11. SEGUNDA BÚSQUEDA OPTUNA REFINADA
# ============================================================

import optuna
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier


# ============================================================
# 11.1 RANGOS REFINADOS POR VARIANTE
# ============================================================

RANGOS_REFINADOS = {

    'mx': {
        'n_estimators': (450, 750),
        'max_depth': (4, 8),
        'learning_rate': (0.02, 0.09),
        'min_child_weight': (1, 3),
        'subsample': (0.60, 0.80),
        'colsample_bytree': (0.65, 0.85),
        'gamma': (2.0, 5.0),
        'reg_alpha': (1e-4, 0.10),
        'reg_lambda': (0.10, 2.0),
        'scale_pos_weight': (1.4, 2.1)
    },

    'es': {
        'n_estimators': (550, 800),
        'max_depth': (2, 5),
        'learning_rate': (0.02, 0.08),
        'min_child_weight': (1, 3),
        'subsample': (0.85, 1.00),
        'colsample_bytree': (0.85, 1.00),
        'gamma': (1.0, 4.0),
        'reg_alpha': (1e-4, 0.05),
        'reg_lambda': (0.50, 8.0),
        'scale_pos_weight': (1.2, 1.8)
    },

    'cu': {
        'n_estimators': (350, 550),
        'max_depth': (2, 5),
        'learning_rate': (0.015, 0.05),
        'min_child_weight': (1, 3),
        'subsample': (0.70, 0.90),
        'colsample_bytree': (0.75, 0.95),
        'gamma': (1.5, 4.0),
        'reg_alpha': (1e-4, 0.10),
        'reg_lambda': (0.05, 1.5),
        'scale_pos_weight': (1.3, 2.0)
    }
}


# ============================================================
# 11.2 FUNCIÓN OBJETIVO REFINADA
# ============================================================

def objective_xgboost_refinado(trial, variante):

    prep = MEJOR_PREP[variante]
    rango = RANGOS_REFINADOS[variante]

    df = cargar_train(
        variante=variante,
        prep=prep
    )

    X = df[
        ['MESSAGE_CLEAN'] + FEATURE_COLS
    ].copy()

    y = df['IS_IRONIC'].astype(int)

    X['MESSAGE_CLEAN'] = (
        X['MESSAGE_CLEAN']
        .fillna('')
        .astype(str)
    )

    X[FEATURE_COLS] = (
        X[FEATURE_COLS]
        .fillna(0)
    )

    params = {

        'n_estimators': trial.suggest_int(
            'n_estimators',
            rango['n_estimators'][0],
            rango['n_estimators'][1],
            step=25
        ),

        'max_depth': trial.suggest_int(
            'max_depth',
            rango['max_depth'][0],
            rango['max_depth'][1]
        ),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            rango['learning_rate'][0],
            rango['learning_rate'][1],
            log=True
        ),

        'min_child_weight': trial.suggest_int(
            'min_child_weight',
            rango['min_child_weight'][0],
            rango['min_child_weight'][1]
        ),

        'subsample': trial.suggest_float(
            'subsample',
            rango['subsample'][0],
            rango['subsample'][1]
        ),

        'colsample_bytree': trial.suggest_float(
            'colsample_bytree',
            rango['colsample_bytree'][0],
            rango['colsample_bytree'][1]
        ),

        'gamma': trial.suggest_float(
            'gamma',
            rango['gamma'][0],
            rango['gamma'][1]
        ),

        'reg_alpha': trial.suggest_float(
            'reg_alpha',
            rango['reg_alpha'][0],
            rango['reg_alpha'][1],
            log=True
        ),

        'reg_lambda': trial.suggest_float(
            'reg_lambda',
            rango['reg_lambda'][0],
            rango['reg_lambda'][1],
            log=True
        ),

        'scale_pos_weight': trial.suggest_float(
            'scale_pos_weight',
            rango['scale_pos_weight'][0],
            rango['scale_pos_weight'][1]
        )
    }

    modelo = XGBClassifier(
        objective='binary:logistic',
        tree_method='hist',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params
    )

    pipeline = Pipeline(
        steps=[
            ('features', crear_preprocesador()),
            ('xgb', modelo)
        ]
    )

    scores = cross_val_score(
        estimator=pipeline,
        X=X,
        y=y,
        cv=CV,
        scoring='f1_macro',
        n_jobs=1
    )

    return scores.mean()


# ============================================================
# 11.3 EJECUTAR SEGUNDA BÚSQUEDA
# ============================================================

N_TRIALS_REFINADOS = 40

studies_refinados = {}

for variante in ['mx', 'es', 'cu']:

    print('\n')
    print('=' * 70)
    print(f'OPTUNA REFINADO - {variante.upper()}')
    print(f'Preprocesamiento: {MEJOR_PREP[variante]}')
    print('=' * 70)

    sampler = optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )

    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        study_name=f'xgboost_refinado_{variante}'
    )

    study.optimize(
        lambda trial: objective_xgboost_refinado(
            trial,
            variante
        ),
        n_trials=N_TRIALS_REFINADOS,
        show_progress_bar=True
    )

    studies_refinados[variante] = study

    print(
        f'\nMejor F1-Macro CV refinado: '
        f'{study.best_value:.4f}'
    )

    print('\nMejores hiperparámetros:')

    for parametro, valor in study.best_params.items():
        print(f'  {parametro}: {valor}')


# ============================================================
# 11.4 COMPARAR PRIMERA VS SEGUNDA BÚSQUEDA
# ============================================================

comparacion_optuna = []

for variante in ['mx', 'es', 'cu']:

    primera = studies[variante]
    segunda = studies_refinados[variante]

    comparacion_optuna.append({
        'variante': variante,

        'f1_primera_busqueda':
            primera.best_value,

        'f1_segunda_busqueda':
            segunda.best_value,

        'mejora':
            segunda.best_value - primera.best_value,

        'mejor_busqueda':
            (
                'segunda'
                if segunda.best_value > primera.best_value
                else 'primera'
            )
    })


df_comparacion_optuna = pd.DataFrame(
    comparacion_optuna
)

print('\n')
print('=' * 70)
print('COMPARACIÓN OPTUNA 1 VS OPTUNA 2')
print('=' * 70)

display(
    df_comparacion_optuna.style.format({
        'f1_primera_busqueda': '{:.4f}',
        'f1_segunda_busqueda': '{:.4f}',
        'mejora': '{:+.4f}'
    })
)


# ============================================================
# 11.5 GUARDAR MEJOR CONFIGURACIÓN DEFINITIVA
# ============================================================

mejores_studies_finales = {}

for variante in ['mx', 'es', 'cu']:

    if (
        studies_refinados[variante].best_value
        >
        studies[variante].best_value
    ):
        mejores_studies_finales[variante] = (
            studies_refinados[variante]
        )
    else:
        mejores_studies_finales[variante] = (
            studies[variante]
        )


# ============================================================
# 11.6 RESUMEN FINAL DE HIPERPARÁMETROS
# ============================================================

resumen_final = []

for variante, study in mejores_studies_finales.items():

    fila = {
        'variante': variante,
        'preprocesamiento':
            MEJOR_PREP[variante],

        'best_f1_macro_cv':
            study.best_value
    }

    fila.update(study.best_params)

    resumen_final.append(fila)


df_mejores_finales = pd.DataFrame(
    resumen_final
)

print('\n')
print('=' * 70)
print('CONFIGURACIÓN DEFINITIVA POR VARIANTE')
print('=' * 70)

display(
    df_mejores_finales.style.format({
        'best_f1_macro_cv': '{:.4f}',
        'learning_rate': '{:.5f}',
        'subsample': '{:.4f}',
        'colsample_bytree': '{:.4f}',
        'gamma': '{:.4f}',
        'reg_alpha': '{:.5f}',
        'reg_lambda': '{:.5f}',
        'scale_pos_weight': '{:.4f}'
    })
)

[I 2026-09-21 17:12:51,823] A new study created in memory with name: xgboost_refinado_mx




OPTUNA REFINADO - MX
Preprocesamiento: normal


  0%|          | 0/40 [00:00<?, ?it/s]

Cargando: ../data/train_clean_mx.csv
[I 2026-09-21 17:12:58,566] Trial 0 finished with value: 0.6060274735770232 and parameters: {'n_estimators': 550, 'max_depth': 8, 'learning_rate': 0.06014196290837443, 'min_child_weight': 2, 'subsample': 0.6312037280884872, 'colsample_bytree': 0.6811989040672406, 'gamma': 2.1742508365045983, 'reg_alpha': 0.0396760507705299, 'reg_lambda': 0.6054365855469247, 'scale_pos_weight': 1.8956508044572318}. Best is trial 0 with value: 0.6060274735770232.
Cargando: ../data/train_clean_mx.csv
[I 2026-09-21 17:13:03,807] Trial 1 finished with value: 0.6131117329853221 and parameters: {'n_estimators': 450, 'max_depth': 8, 'learning_rate': 0.06995068079750541, 'min_child_weight': 1, 'subsample': 0.6363649934414202, 'colsample_bytree': 0.6866809019706868, 'gamma': 2.912726728878613, 'reg_alpha': 0.0037520558551242854, 'reg_lambda': 0.364731628491121, 'scale_pos_weight': 1.6038603981386292}. Best is trial 1 with value: 0.6131117329853221.
Cargando: ../data/train_cle

[I 2026-09-21 17:16:32,105] A new study created in memory with name: xgboost_refinado_es


...

  0%|          | 0/40 [00:00<?, ?it/s]

Cargando: ../data/train_clean_stem_cu.csv
[I 2026-09-21 17:21:55,316] Trial 0 finished with value: 0.660488321764827 and parameters: {'n_estimators': 425, 'max_depth': 5, 'learning_rate': 0.03621056763954017, 'min_child_weight': 2, 'subsample': 0.7312037280884873, 'colsample_bytree': 0.7811989040672406, 'gamma': 1.6452090304204987, 'reg_alpha': 0.0396760507705299, 'reg_lambda': 0.3862689194653677, 'scale_pos_weight': 1.795650804457232}. Best is trial 0 with value: 0.660488321764827.
Cargando: ../data/train_clean_stem_cu.csv
[I 2026-09-21 17:22:01,749] Trial 1 finished with value: 0.6608323155450291 and parameters: {'n_estimators': 350, 'max_depth': 5, 'learning_rate': 0.040865594623548177, 'min_child_weight': 1, 'subsample': 0.73636499344142, 'colsample_bytree': 0.7866809019706867, 'gamma': 2.260605607398844, 'reg_alpha': 0.0037520558551242854, 'reg_lambda': 0.21727270548647376, 'scale_pos_weight': 1.5038603981386294}. Best is trial 1 with value: 0.6608323155450291.
Cargando: ../data/t

,variante,f1_primera_busqueda,f1_segunda_busqueda,mejora,mejor_busqueda
0,mx,0.6218,0.6259,+0.0041,segunda
1,es,0.7132,0.7135,+0.0004,segunda
2,cu,0.6684,0.6743,+0.0059,segunda




CONFIGURACIÓN DEFINITIVA POR VARIANTE


,variante,preprocesamiento,best_f1_macro_cv,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,scale_pos_weight
0,mx,normal,0.6259,475,6,0.02106,3,0.6518,0.7825,2.9351,0.00363,0.51438,1.5294
1,es,lemma,0.7135,775,4,0.02954,1,0.9586,0.8837,2.8417,0.04119,6.86735,1.2703
2,cu,stem,0.6743,450,3,0.03169,1,0.7817,0.7951,3.7067,0.00475,0.35253,1.5151


In [5]:
# ============================================================
# RECUPERAR OPTUNA REFINADO SOLO PARA CU
# ============================================================

import optuna

variante = 'cu'

sampler = optuna.samplers.TPESampler(
    seed=RANDOM_STATE
)

study_cu = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    study_name='xgboost_refinado_cu_recuperado'
)

study_cu.optimize(
    lambda trial: objective_xgboost_refinado(
        trial,
        variante
    ),
    n_trials=40,
    show_progress_bar=True
)

print('\nMejor F1-Macro CU:')
print(study_cu.best_value)

print('\nParámetros exactos CU:')

for parametro, valor in study_cu.best_params.items():
    print(f'{parametro}: {repr(valor)}')

[I 2026-09-21 17:25:40,547] A new study created in memory with name: xgboost_refinado_cu_recuperado


  0%|          | 0/40 [00:00<?, ?it/s]

Cargando: ../data/train_clean_stem_cu.csv
[I 2026-09-21 17:25:47,703] Trial 0 finished with value: 0.660488321764827 and parameters: {'n_estimators': 425, 'max_depth': 5, 'learning_rate': 0.03621056763954017, 'min_child_weight': 2, 'subsample': 0.7312037280884873, 'colsample_bytree': 0.7811989040672406, 'gamma': 1.6452090304204987, 'reg_alpha': 0.0396760507705299, 'reg_lambda': 0.3862689194653677, 'scale_pos_weight': 1.795650804457232}. Best is trial 0 with value: 0.660488321764827.
Cargando: ../data/train_clean_stem_cu.csv
[I 2026-09-21 17:25:53,907] Trial 1 finished with value: 0.6608323155450291 and parameters: {'n_estimators': 350, 'max_depth': 5, 'learning_rate': 0.040865594623548177, 'min_child_weight': 1, 'subsample': 0.73636499344142, 'colsample_bytree': 0.7866809019706867, 'gamma': 2.260605607398844, 'reg_alpha': 0.0037520558551242854, 'reg_lambda': 0.21727270548647376, 'scale_pos_weight': 1.5038603981386294}. Best is trial 1 with value: 0.6608323155450291.
Cargando: ../data/t